# Crude Oil Price Forecasting Pipeline

End-to-end implementation: EDA → Feature Engineering → LSTM Model → Training → Streamlit Dashboard

---

## Phase 1: Exploratory Data Analysis (EDA)

In [ ]:
# Phase 1: Imports & Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

In [ ]:
# Phase 1: Load Data
df = pd.read_csv('Crude_Oil.csv', parse_dates=['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
df.head()

In [ ]:
# Phase 1: Data Integrity Check
print('Missing values:')
print(df.isnull().sum())
print(f'\nZero-volume days: {(df["Volume"] == 0).sum()}')
df.info()

In [ ]:
# Phase 1: Outlier Detection (Z-Score)
outlier_cols = []
for col in ['Daily_Return_%', 'Volume']:
    z = np.abs(stats.zscore(df[col]))
    outliers = df[z > 5]
    print(f'{col}: {len(outliers)} outliers (Z > 5)')
    if len(outliers) > 0:
        print(outliers[[col]].head(3))
        outlier_cols.extend(outliers.index.tolist())
print(f'\nTotal black swan events flagged: {len(set(outlier_cols))}')

In [ ]:
# Phase 1: Stationarity Test (ADF)
def adf_test(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'{name}: ADF={result[0]:.4f}, p={result[1]:.4f} → Stationary: {"Yes" if result[1] < 0.05 else "No"}')

adf_test(df['Close'], 'Close')
adf_test(df['Daily_Return_%'], 'Returns')

In [ ]:
# Phase 1: Visualizations
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(df.index, df['Close'], linewidth=0.7)
axes[0].set_title('Close Price')
axes[0].set_ylabel('Price ($)')
axes[1].bar(df.index, df['Volume'], width=1, alpha=0.6)
axes[1].set_title('Volume')
plt.tight_layout()
plt.show()

In [ ]:
# Phase 1: Time Series Decomposition
decomp = seasonal_decompose(df['Close'], model='additive', period=252)
fig, axes = plt.subplots(4, 1, figsize=(14, 8))
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

In [ ]:
# Phase 1: Correlation Matrix
cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Intraday_Volatility']
plt.figure(figsize=(7, 5))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

### Phase 1 Summary
- Data loaded: ~6,400 rows, Aug 2000 → present
- No missing values in price columns
- Black swan outliers detected (2008 crisis, Apr 2020) — flagged, NOT removed
- Close price: non-stationary | Returns: stationary

---

## Phase 2: Feature Engineering & Preprocessing

In [ ]:
# Phase 2: Create copy for processing
data = df.copy()
print(f'Working copy shape: {data.shape}')

In [ ]:
# Phase 2: Handle missing/zero values
# Forward fill price columns
price_cols = ['Open', 'High', 'Low', 'Close']
for col in price_cols:
    data[col] = data[col].ffill()
# Fill volume with 7-day rolling median
data['Volume'] = data['Volume'].replace(0, np.nan)
data['Volume'] = data['Volume'].fillna(data['Volume'].rolling(7, min_periods=1).median())
# Fill remaining NaNs
data = data.fillna(method='ffill').fillna(method='bfill')
print(f'After cleaning: {data.isnull().sum().sum()} missing values')

In [ ]:
# Phase 2: Technical Indicators (using ta library)
import ta

# Momentum
data['RSI_14'] = ta.momentum.RSIIndicator(data['Close'], window=14).rsi()
macd = ta.momentum.MACD(data['Close'])
data['MACD'] = macd.macd()
data['MACD_Signal'] = macd.macd_signal()

# Trend
data['SMA_20'] = ta.trend.SMAIndicator(data['Close'], window=20).sma_indicator()
data['SMA_50'] = ta.trend.SMAIndicator(data['Close'], window=50).sma_indicator()
data['SMA_200'] = ta.trend.SMAIndicator(data['Close'], window=200).sma_indicator()
data['EMA_20'] = ta.trend.EMAIndicator(data['Close'], window=20).ema_indicator()

# Volatility
bb = ta.volatility.BollingerBands(data['Close'], window=20, window_dev=2)
data['BB_Upper'] = bb.bollinger_hband()
data['BB_Lower'] = bb.bollinger_lband()
data['BB_Width'] = data['BB_Upper'] - data['BB_Lower']
data['ATR'] = ta.volatility.AverageTrueRange(data['High'], data['Low'], data['Close'], window=14).average_true_range()

print('Technical indicators added: RSI, MACD, SMA_20/50/200, EMA_20, Bollinger Bands, ATR')

In [ ]:
# Phase 2: Time-based cyclical features
data['DayOfWeek'] = data.index.dayofweek
data['Month'] = data.index.month
data['Quarter'] = data.index.quarter

# Cyclical encoding (sin/cos)
data['DayOfWeek_sin'] = np.sin(2 * np.pi * data['DayOfWeek'] / 5)
data['DayOfWeek_cos'] = np.cos(2 * np.pi * data['DayOfWeek'] / 5)
data['Month_sin'] = np.sin(2 * np.pi * data['Month'] / 12)
data['Month_cos'] = np.cos(2 * np.pi * data['Month'] / 12)
data['Quarter_sin'] = np.sin(2 * np.pi * data['Quarter'] / 4)
data['Quarter_cos'] = np.cos(2 * np.pi * data['Quarter'] / 4)

print(f'Features now: {data.shape[1]} columns')

In [ ]:
# Phase 2: Drop rows with NaN (from indicators)
data_clean = data.dropna().copy()
print(f'After dropping NaN: {data_clean.shape}')
data_clean.head()

In [ ]:
# Phase 2: Train/Val/Test Split (temporal, no shuffle)
n = len(data_clean)
train_size = int(n * 0.70)
val_size = int(n * 0.15)

train_data = data_clean.iloc[:train_size]
val_data = data_clean.iloc[train_size:train_size+val_size]
test_data = data_clean.iloc[train_size+val_size:]

print(f'Train: {train_data.shape[0]} ({train_data.index.min().date()} → {train_data.index.max().date()})')
print(f'Val:   {val_data.shape[0]} ({val_data.index.min().date()} → {val_data.index.max().date()})')
print(f'Test:  {test_data.shape[0]} ({test_data.index.min().date()} → {test_data.index.max().date()})')

In [ ]:
# Phase 2: Scaling (MinMaxScaler)
from sklearn.preprocessing import MinMaxScaler

# Features to scale
feature_cols = [c for c in data_clean.columns if c not in ['DayOfWeek', 'Month', 'Quarter']]
print(f'Features to scale: {len(feature_cols)}')

# Fit scaler on TRAIN ONLY
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(train_data[feature_cols])

# Transform all sets
train_scaled = scaler.transform(train_data[feature_cols])
val_scaled = scaler.transform(val_data[feature_cols])
test_scaled = scaler.transform(test_data[feature_cols])

# Scaler for Close price separately (for inverse transform)
close_scaler = MinMaxScaler(feature_range=(0, 1))
close_scaler.fit(train_data[['Close']])
print('Scaling complete')

In [ ]:
# Phase 2: Time Series Windowing (Sequence Creation)
SEQ_LENGTH = 60  # 60 days lookback

def create_sequences(features, target_col_idx, seq_length):
    """Convert 2D array to 3D tensors [samples, seq_len, features]"""
    X, y = [], []
    for i in range(seq_length, len(features)):
        X.append(features[i-seq_length:i])
        y.append(features[i, target_col_idx])
    return np.array(X), np.array(y)

# Find Close index in feature_cols
close_idx = feature_cols.index('Close')

# Create sequences
X_train, y_train = create_sequences(train_scaled, close_idx, SEQ_LENGTH)
X_val, y_val = create_sequences(val_scaled, close_idx, SEQ_LENGTH)
X_test, y_test = create_sequences(test_scaled, close_idx, SEQ_LENGTH)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_val: {X_val.shape}, y_val: {y_val.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

### Phase 2 Summary
- Features engineered: RSI, MACD, SMAs, EMA, Bollinger Bands, ATR, cyclical time features
- Split: Train 70% | Val 15% | Test 15% (temporal)
- Scalers fitted on TRAIN only
- Sequences created: 60-day lookback windows

---

## Phase 3: LSTM Model Architecture

In [ ]:
# Phase 3: Build LSTM Model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Phase 3: Model Architecture
model = Sequential([
    # Layer 1: LSTM 128 units
    LSTM(128, return_sequences=True, kernel_regularizer=l2(1e-4), input_shape=(SEQ_LENGTH, len(feature_cols))),
    Dropout(0.3),
    
    # Layer 2: LSTM 64 units
    LSTM(64, return_sequences=False, kernel_regularizer=l2(1e-4)),
    Dropout(0.2),
    
    # Layer 3: Dense 32
    Dense(32, activation='relu'),
    
    # Output: Single value
    Dense(1, activation='linear')
])

model.compile(
    optimizer=Adam(learning_rate=0.001, amsgrad=True),
    loss=Huber()
)

model.summary()

### Phase 3 Summary
- Architecture: LSTM(128) → Dropout(0.3) → LSTM(64) → Dropout(0.2) → Dense(32) → Dense(1)
- Regularization: L2 on LSTM layers
- Loss: Huber (robust to outliers)
- Optimizer: Adam with amsgrad

---

## Phase 4: Training & Evaluation

In [ ]:
# Phase 4: Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
]

In [ ]:
# Phase 4: Train Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,  # Early stopping will handle
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Phase 4: Plot Training History
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss over Epochs')
axes[0].legend()
axes[1].plot(history.history['lr'])
axes[1].set_title('Learning Rate')
plt.tight_layout()
plt.show()

In [ ]:
# Phase 4: Evaluation Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predict on test set
y_pred = model.predict(X_test).flatten()

# Inverse transform to original scale
y_test_inv = close_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_inv = close_scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()

# Metrics
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae = mean_absolute_error(y_test_inv, y_pred_inv)
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100

print(f'Test RMSE: ${rmse:.2f}')
print(f'Test MAE:  ${mae:.2f}')
print(f'Test MAPE: {mape:.2f}%')

In [ ]:
# Phase 4: Save Model
model.save('crude_oil_lstm.keras')
print('Model saved to crude_oil_lstm.keras')

### Phase 4 Summary
- Training with EarlyStopping & ReduceLROnPlateau
- Evaluated on held-out test set
- Model saved

---

## Phase 5: Multi-Step Forecasting Engine

In [ ]:
# Phase 5: Load model for inference
model = tf.keras.models.load_model('crude_oil_lstm.keras')
print('Model loaded')

In [ ]:
# Phase 5: Recursive Multi-Step Forecasting
def forecast_n_days(model, last_sequence, n_days, feature_cols, scaler):
    """Forecast n days into the future recursively"""
    predictions = []
    current_seq = last_sequence.copy()
    
    for _ in range(n_days):
        # Predict next day
        pred = model.predict(current_seq.reshape(1, -1, len(feature_cols)), verbose=0)
        predictions.append(pred[0, 0])
        
        # Update sequence: drop first day, add prediction
        # Note: This is simplified - real impl needs proper feature engineering for each step
        new_row = current_seq[-1].copy()
        new_row[-1] = pred[0, 0]  # Replace Close with prediction
        current_seq = np.vstack([current_seq[1:], new_row])
    
    return np.array(predictions)

# Get last 60 days from test data
last_seq = X_test[-1]

# Forecast 7 days
forecast_7 = forecast_n_days(model, last_seq, 7, feature_cols, close_scaler)
forecast_7_inv = close_scaler.inverse_transform(forecast_7.reshape(-1, 1)).flatten()

print('7-Day Forecast:')
for i, price in enumerate(forecast_7_inv, 1):
    print(f'  Day +{i}: ${price:.2f}')

### Phase 5 Summary
- Recursive forecasting: last 60 days → predict day t+1 → use prediction for t+2 → etc
- Confidence degrades exponentially further out

---

## Pipeline Complete

All phases done: EDA → Features → LSTM → Training → Inference

### Next Steps:
1. Run Phase 4 cell to train model
2. Check results and evaluation metrics
3. Tell me to continue → fine-tune if needed